In [4]:
import pathlib as pl
import pickle as pck
import os

import pandas as pd

cfg_nb = pl.Path("../../load-config.ipynb").resolve(strict=True)
%run $cfg_nb

_NB_SESSION = __session__
NB_NAME = pl.Path(_NB_SESSION).name
NB_PATH = pl.Path(_NB_SESSION).parent
NB_REL_PATH = NB_PATH.relative_to(CONFIG["project_repo"]).joinpath(NB_NAME)

FORCE_REBUILD_CACHE = True

cache_file_path = prep_cache_file(NB_REL_PATH, "verkko_sex_chrom_arhie_v2.pkl")
if not cache_file_path.is_file() or FORCE_REBUILD_CACHE:

    fasta_files = []
    for chrY_asm_subfolder in CONFIG["hilbert_verkko_sub_folder_arhie"]:
    
        DATA_ROOT = CONFIG["local_hilbert_prefix"].joinpath(chrY_asm_subfolder)

        assert DATA_ROOT.is_dir()
    
        toplevel_glob = DATA_ROOT.glob("*.fa.gz")
        
        for fasta_file in toplevel_glob:
            file_size = os.stat(fasta_file).st_size
            if file_size == 0:
                continue
            fasta_files.append(fasta_file)

    reflevel_glob = CONFIG["local_hilbert_prefix"].joinpath(
        CONFIG["hilbert_extracted_seq_workdir"],
        "inject-ref"
    ).resolve()
    reflevel_glob.resolve(strict=True)
    if not reflevel_glob.is_dir():
        print("warning: ref injection folder does not exist")
    else:
        reflevel_glob = reflevel_glob.glob("**/*.fasta")
        for fasta_file in reflevel_glob:
            if "chrX" in fasta_file.name:
                continue
            fasta_files.append(fasta_file)
    
    cache_dump = {
        "fasta_files": fasta_files
    }
    with open(cache_file_path, "wb") as cache:
        _ = pck.dump(cache_dump, cache)
else:
    with open(cache_file_path, "rb") as cache:
        cache_dump = pck.load(cache)
        fasta_files = cache_dump["fasta_files"]

sample_sheet = []
for fasta_file in fasta_files:
    if "_chrY" in fasta_file.name:
        sample = fasta_file.name.rsplit("_", 1)[0]
    else:
        sample = fasta_file.name.rsplit(".", 2)[0]
    remote_path = replace_path_prefix(fasta_file)
    sex = "male"

    sample_group = fasta_file.parent.parent.stem
    if sample_group in ["hgsvc", "hprc", "ceph"]:
        pass
    else:
        sample_group = fasta_file.parent.stem
        if sample_group == "inject-ref":
            sample_group = "REF"
        else:
            raise ValueError(f"Unk. sample group: {fasta_file}")        
    sample_sheet.append((sample, sex, sample_group, remote_path))
    

sample_sheet = pd.DataFrame.from_records(
    sample_sheet,
    columns=["sample", "sample_sex", "sample_group", "input_path"]
)
sample_sheet.set_index("sample", inplace=True)
sample_sheet.sort_index(inplace=True)

sample_sheet_file = CONFIG["project_repo"].joinpath("samples", "verkko_chrY_arhie_freeze.tsv")
sample_sheet_file.parent.mkdir(exist_ok=True, parents=True)

with open(sample_sheet_file, "w") as table:
    _ = table.write(f"# {TIMESTAMP}\n")
    _ = table.write(f"# N={sample_sheet.shape[0]}\n")
    for group, count in sample_sheet["sample_group"].value_counts().items():
        _ = table.write(f"# group {group}: N={count}\n")
    sample_sheet.to_csv(table, sep="\t", header=True, index=True)


